This will have the same content as the single-run kmeans file but in a jupyter notebook format it'll be easier to run and manage 

In [32]:
import numpy as np 
import matplotlib.pyplot as plt 
import torch
import csv
import pandas as pd  
from scipy import stats as st
from sklearn.preprocessing import MinMaxScaler
from collections import Counter 
import tqdm

events = 10 
density = '1' 
noise = 0 
filename = None 
folder = None 
k = 100
max_iters = 50

In [52]:
# def kmeans_gpu(data, k, max_iters=100):
#     data = data.to(device)
#     centroids = data[torch.randperm(len(data))[:k]]

#     for i in range(max_iters):
#         #print(f"Iteration: {i+1}")
#         dist = torch.cdist(data, centroids) 
#         ai_labels = torch.argmin(dist, dim=1)

#         # update centroids 
#         for j in range(k):
#             centroids[j] = data[ai_labels == j].mean(dim=0)

#     return ai_labels, centroids
#     # return ai_labels.cpu().numpy(), centroids.numpy()

# device = "cuda" if torch.cuda.is_available() else "cpu" 
def kmeans(data, k, max_iters=100, tol=1e-4):
    np.random.seed(42)
    random_indices = np.random.choice(len(data), size=k, replace=False)
    centroids = data[random_indices]

    for i in range(max_iters):
        distances = np.linalg.norm(data[:, np.newaxis] - centroids, axis=2)
        labels = np.argmin(distances, axis=1)

        new_centroids = []
        for j in range(k):
            cluster_points = data[labels == j]
            if len(cluster_points) == 0:
                # Handle empty cluster by reinitializing to a random point
                new_centroid = data[np.random.choice(len(data))]
            else:
                new_centroid = cluster_points.mean(axis=0)
            new_centroids.append(new_centroid)

        new_centroids = np.array(new_centroids)

        if np.linalg.norm(new_centroids - centroids) < tol:
            break
        centroids = new_centroids

    return labels, centroids


In [53]:
def labelmaker(events=None, sp_density=None, t_density=None, noise=None, filename = None, folder = None): 
    '''creates labels based on my naming convention for different files, keeps it consistent and easy'''
    if folder: 
        folder = folder + '/'

    if filename: 
        datafile = str(filename) 
    else: 
        datafile = str(events) + 'ev_' + str(sp_density) + 'spd_' + str(t_density) + 'td_n' + str(noise)
    

    labelfile = 'labels_' + datafile + '.csv'
    sourcefile = 'sources_' + datafile + '.csv'
    ai_labelfile = datafile + '_results' + '.csv'
    centroidfile = datafile + '_centroids' + '.csv'
    datafile = datafile + '.csv'

    if folder: 
        datafile = folder + datafile 
        centroidfile = folder + centroidfile 
        ai_labelfile = folder + ai_labelfile 
        labelfile = folder + labelfile 
        sourcefile = folder + sourcefile 

    
    return datafile, labelfile, sourcefile, ai_labelfile, centroidfile

### READFILES for labels and data written out in 3d columns
def readfiles(datafile, labelfile, sourcefile): 
    '''data reader and simplifier for files that haven't been passed through the algorithm'''
    columns = ['x[px]', 'y[px]', 't[s]']
    
    dataread = pd.read_csv(datafile) 
    data = np.array(dataread[columns])
    
    labelread = pd.read_csv(labelfile)
    labels = np.array(labelread['labels'])

    sourceread = pd.read_csv(sourcefile) 
    sources = np.array(sourceread[columns])

    return data, labels, sources

### READLINES for labels and data written out one event and label per row
def readlines(datafile, labelfile, sourcefile):
    columns = ['x[px]', 'y[px]', 't[s]']
    # read in source file 
    sourceread = pd.read_csv(sourcefile)
    # turn into an array 
    sources = np.array(sourceread[columns])
    # read in datafile 
    # read in labelfile 
    # initialize data array
    data_arr = np.array([])
    # initialize label array
    label_arr = np.array([[]])
    with open(datafile, newline = '') as dataf, open(labelfile, newline = '') as labelf: 
        datareader = csv.reader(dataf)
        labelreader = csv.reader(labelf)

        next(labelreader)

        for datarow, labelrow in zip(datareader, labelreader): 
            templabel = [float(labelrow[0])]*(int(len(datarow)/3))
            templabel = np.array(templabel)
            labels = templabel.reshape(-1, 1)
            label_arr = np.append(label_arr, labels)
            tempdata = np.array(datarow, dtype = float)
            data_arr = np.append(data_arr, tempdata)

    data_arr = data_arr.reshape(-1,3)
    print(len(data_arr) == len(label_arr))

    return data_arr, label_arr, sources 




def readai(datafile, labelfile, sourcefile, ai_labelfile, centroidfile):
    '''file reader and data simplifier for data thats been through the algorithm'''
    columns = ['x[px]', 'y[px]', 't[s]']
    
    dataread = pd.read_csv(datafile) 
    data = np.array(dataread[columns])
    
    labelread = pd.read_csv(labelfile)
    labels = np.array(labelread['labels'])

    sourceread = pd.read_csv(sourcefile) 
    sources = np.array(sourceread[columns])

    ai_labelsread = pd.read_csv(ai_labelfile)
    ai_labels = np.array(ai_labelsread['labels'])

    centroidread = pd.read_csv(centroidfile) 
    centroids = np.array(centroidread[columns])

    return data, labels, sources, ai_labels, centroids

In [54]:
datafile, labelfile, sourcefile, ai_labelfile, centroidfile = labelmaker(filename = '100ev_n0_es100', folder = 'LSTM')
data, labels, sources = readlines(datafile, labelfile, sourcefile)

True


Generating datafile names and data + normalizing it using MinMaxScaling below: 

In [55]:
datafile, labelfile, sourcefile, ai_labelfile, centroidfile = labelmaker(filename='100ev_n0_es100', folder='LSTM')
# read out the files, readiles for kmeans formatted data, readlines for LSTM formatted data (more info above in function definition)
data, labels, sources = readlines(datafile, labelfile, sourcefile)
# establish feature range and transform data, can comment this out to turn off scaling, naming is the same
scaler = MinMaxScaler(feature_range=(0,1))
# scale coordinate data 
data = scaler.fit_transform(data)
sources = scaler.fit_transform(sources)
# turn data into a tensor 
# data_tensor = torch.Tensor(data)

True


Running k-means algorithm

In [56]:
# ai_labels, centroids = kmeans_gpu(data_tensor, k, max_iters)
ai_labels, centroids = kmeans(data, k, max_iters)

Writing to the centroid and ai_label files 

In [84]:
with open(ai_labelfile, mode = 'w', newline='') as wfile: 
    writer = csv.writer(wfile)
    writer.writerow(['labels'])
    for item in ai_labels: 
                writer.writerow([item])

columns = ['x[px]', 'y[px]', 't[s]']
with open(centroidfile, mode = 'w', newline = '') as wfile: 
    writer = csv.writer(wfile)
    writer.writerow(columns)
    writer.writerows(centroids)

In [7]:
# # reading out ai generated files after the algorithm run 

# data, labels, sources, ai_labels, centroids = readai(datafile, labelfile, sourcefile, ai_labelfile, centroidfile)

Breaks and sizing functions for the true label-focused loss function 

In [85]:
def breaks(array): 
    '''This function takes an array, goes through it item by item, and returns the list of indices where the value changes'''
    value = array[0] 
    indices = []
    for index, ele in enumerate(array): 
        if ele != value:
            indices.append(index)
            value = array[index]
    indices.append(len(array))

    return indices

def sizes(indices): 
    '''This function takes a list of indices and calculates the number of items belonging to each value by taking the difference between 
    subsequent indices'''
    gaps = []
    prev=0
    for i in range(len(indices)):
        gaps.append(indices[i]-prev)
        prev = indices[i]

    return gaps

True Label-Focused Loss, ripped from the losses file 

In [86]:
# additional function needed for the loss function 
def modded_mode(array): 
    '''This function will return event labels, its a modification on a normal mode function where the label has to appear at least 33% of the time to be a label in that chunk.'''
    n = len(array)
    if n == 0: 
        return []
    
    threshold = n/3 
    counts = Counter(array)

    result = [key for key, count in counts.items() if count >= threshold]
    return result 

# THE loss func. 

def truth_based_loss(true_labels, ai_labels):
    # labels are already sorted by true labels predictions 

    # getting break indices and cluster sizes 
    break_indices = breaks(true_labels)
    gaps = sizes(break_indices)
    n_events = len(gaps)

    # initialize variables 
    counter = Counter()
    fractions_misIDs = []
    total_splits, ev_per_split = 0,0 
    total_splits = 0 
    total_misIDs = 0 


    # process each chunk that's separated by truth gaps 
    for start, end, gap in zip([0] + break_indices[:-1], break_indices, gaps):
        chunk = ai_labels[start:end]
        chunk_modes = modded_mode(chunk)

        # update counter
        counter.update(chunk_modes)
    
        e_in_split = len(chunk_modes) # number of ai events found in the chunk
        if e_in_split > 1: 
            total_splits += 1
            ev_per_split += e_in_split # if there is more than one, increase splits and events in split 
        
        misIDs = sum(1 for item in chunk if item not in chunk_modes)
        total_misIDs += misIDs
        fractions_misIDs.append(misIDs/gap)
    
    # Combination Statistics 
    repeat_labels = {k: v for k, v in counter.items() if v>1}
    total_combos = len(repeat_labels) # the amount of combinations is the same as the amount of labels that get repeated through the set 
    ev_per_combo = sum(repeat_labels.values())/total_combos if total_combos else 0 # sum all repeats together and average over number of combinations 
    frac_combos = total_combos / n_events # fraction of events experiencing combination
    
    # other stats 
    ev_per_split = ev_per_split/total_splits if total_splits else 0
    frac_splits = total_splits/n_events
    avg_misIDs = np.mean(fractions_misIDs)

    # output da resultssss
    # print(f"The fraction of splits over all events is {frac_splits}")
    # print(f"The average number of events involved in a single split is {ev_per_split}")
    # print(f"The fraction of combinations over all events is {frac_combos}")
    # print(f"The average number of events involved in a single combo is {ev_per_combo}")
    # print(f"The average fraction of photons misidentified in each event is {avg_misIDs}")

    return frac_splits, ev_per_split, frac_combos, ev_per_combo, avg_misIDs

AI Label-Focused Loss function

In [87]:
# these results should match up exactly with the previous loss function within rounding errors 

import numpy as np
from collections import Counter

def ai_based_loss(true_labels, network_labels):
    """
    More efficient implementation of the AI-based loss function.
    This function clusters AI labels and calculates relevant loss statistics.
    """
    
    # Sort labels based on AI predictions
    sorted_pairs = sorted(zip(network_labels, true_labels))
    reo_network_labels, reo_true_labels = zip(*sorted_pairs)

    # Get break indices and cluster sizes
    break_indices = breaks(reo_network_labels)
    gaps = sizes(break_indices)
    # n_events = len(gaps)

    # Initialize variables
    counter = Counter()
    fractions_misIDs = []
    total_combos, ev_per_combo = 0, 0
    total_splits = 0
    total_misIDs = 0 

    # Process each AI-clustered chunk
    for start, end, gap in zip([0] + break_indices[:-1], break_indices, gaps):
        chunk = reo_true_labels[start:end]
        chunk_modes = modded_mode(chunk)  # Get dominant modes (max 3)
        
        # Update Counter directly  
        counter.update(chunk_modes)

        e_in_combo = len(chunk_modes)
        if e_in_combo > 1:
            total_combos += 1
            ev_per_combo += e_in_combo
        
        misIDs = sum(1 for item in chunk if item not in chunk_modes)
        total_misIDs += misIDs
        fractions_misIDs.append(misIDs / gap)
    
    n_events = len(counter)
    # Compute split statistics
    repeat_labels = {k: v for k, v in counter.items() if v > 1}
    total_splits = len(repeat_labels)
    ev_per_split = sum(repeat_labels.values()) / total_splits if total_splits else 0
    frac_splits = total_splits / n_events

    # Compute final metrics
    ev_per_combo = ev_per_combo / total_combos if total_combos else 0
    frac_combos = total_combos / n_events
    avg_misIDs = np.mean(fractions_misIDs)


    # Output results
    print(f"The fraction of splits over all events is {frac_splits}")
    print(f"The average number of events involved in a single split is {ev_per_split}")
    print(f"The fraction of combinations over all events is {frac_combos}")
    print(f"The average number of events involved in a single combo is {ev_per_combo}")
    print(f"The average fraction of photons misidentified in each event is {avg_misIDs}")

    return frac_splits, ev_per_split, frac_combos, ev_per_combo, avg_misIDs




Running Loss functions

In [88]:
# testing to compare ai and truth based loss functions 
print("---- TRUTH BASED LOSS: ----")
truth_based_loss(labels, ai_labels)
print("---- AI BASED LOSS: ----")
ai_based_loss(labels, ai_labels)



---- TRUTH BASED LOSS: ----
---- AI BASED LOSS: ----
The fraction of splits over all events is 0.25
The average number of events involved in a single split is 2.35
The fraction of combinations over all events is 0.1
The average number of events involved in a single combo is 2.0
The average fraction of photons misidentified in each event is 0.0420843991782307


(0.25, 2.35, 0.1, 2.0, 0.0420843991782307)

In [12]:
# # test dataset for loss function testing 

# true_labelz = np.array([1,1,1,1,2,2,3,3,3,3,3,3])
# ai_labelz = np.array([8,55,55,55,55,55,101,101,8,8,55,8])

# truth_based_loss(true_labelz, ai_labelz)
# print("----")
# ai_based_loss(true_labelz, ai_labelz)

# # (8, 8, 8, 8, 55, 55, 55, 55, 55, 55, 101, 101)
# # (1, 3, 3, 3, 1, 1, 1, 2, 2, 3, 3, 3)

In [13]:
# ELBOW METHOD 
def elbowmeth(int_range, datafile, folder=None, iterations=None, pltfile=None, pltnum=None, weighted = True):
    '''will perform the elbow method on a given dataset for a given range of k integers and return the k value with the lowest loss, as well as the value of that loss
    and a plot demonstrating the loss'''
    datafile, labelfile, sourcefile, ai_labelfile, centroidfile = labelmaker(filename = datafile, folder = folder)
    data, labels, sources = readfiles(datafile, labelfile, sourcefile)
    scaler = MinMaxScaler(feature_range=(0,1))
    # scale coordinate data 
    data = scaler.fit_transform(data)
    sources = scaler.fit_transform(sources)
    # turn data into a tensor 
    data_tensor = torch.Tensor(data)
    total_losses = []    

    coeff = [1,2,3] if weighted else [1,1,1]

    for kval in int_range: 
        ai_labels, centroids = kmeans_gpu(data_tensor, k=kval, max_iters=iterations)
        frac_splits, ev_per_split, frac_combos, ev_per_combo, avg_misIDs = truth_based_loss(true_labels = labels, ai_labels=ai_labels)
        if weighted: 
            coeff = [1,2,3]
        total_loss = np.dot(coeff, [avg_misIDs, frac_splits, frac_combos])
        total_losses.append(total_loss)

    if pltnum%5 == 0:
        #generate plot for every fifth iteration
        plt.figure()
        plt.plot(int_range, total_losses)
        plt.xlabel("K values")
        plt.ylabel("Total loss")
        plt.title(f"Elbow method for k = {int_range} for {datafile} {pltnum}")
        plt.tight_layout()
        plt.savefig(pltfile)   
        print(f"{pltnum}% complete!")     

    min_loss = np.min(total_losses)
    min_k = int_range[total_losses.index(min_loss)]

    print(f"The minimum loss is {min_loss} with a k value of {min_k}")
    return min_loss, min_k


In [14]:
from segmentation_sim import new_parallel_sim # this only works if parsed arguments in segmentation_sim.py are commented out!

In [15]:
import sys
import os

# Suppress stdout temporarily
class SuppressPrints:
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')

    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout.close()
        sys.stdout = self._original_stdout

In [ ]:
#big algo 

k_range = [95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105]
bigLossFile = '0.5temp100x100/UWloss_totals.csv'
columns = ['lowest kval', 'min loss']


with open(bigLossFile, mode = 'w', newline = '') as wfile: 
    writer = csv.writer(wfile)
    writer.writerow(columns) # write the columns into the loss file to begin with 

for i in range(100): # 100 iterations to get good statistics 
    with SuppressPrints():
        new_parallel_sim(100, 0, 10, t_density = '.5', folder = '0.5temp100x100', dataSaveID = 'UWtemp') # generate data of 100 events, no noise, full density, using 5 cores for speed 
    
    print(f"{i}th datafile generated!")
    min_loss, min_k = elbowmeth(k_range, datafile = 'UWtemp', folder = '0.5temp100x100', iterations = 100, pltfile = f'0.5temp100x100/UWkvals_losses_{i}.png', pltnum = i, weighted = False) # run elbow meth!
    print(f"{i}th losses and k values are: {min_loss}, {min_k}")
    with open(bigLossFile, mode = 'a', newline = '') as wfile: # write in the best loss and best kval 
        writer = csv.writer(wfile)
        writer.writerow([min_k, min_loss])

In [ ]:
# read out the two temp 100x100 files 
# plot histogram 
# calculate averages and stds

filenameW = 'temp100x100/loss_totals.csv'
columns = ['lowest kval', 'min loss']
datareadW = pd.read_csv(filenameW)
Wdata = np.array(datareadW[columns])

filenameUW = 'temp100x100/UWloss_totals.csv'
datareadUW = pd.read_csv(filenameUW)
UWdata = np.array(datareadUW[columns])

avg_W_loss = np.mean(Wdata[:,1])
std_W_loss = np.std(Wdata[:,1])
avg_W_k = np.mean(Wdata[:,0])
std_W_k = np.std(Wdata[:,0])

avg_UW_loss = np.mean(UWdata[:,1])
std_UW_loss = np.std(UWdata[:,1])
avg_UW_k = np.mean(UWdata[:,0])
std_UW_k = np.std(UWdata[:,0])

with open(file = 'temp100x100/results.csv', mode = 'w') as wfile: 
    writer = csv.writer(wfile)
    writer.writerow(["value", "avg", "std"])
    writer.writerow(["W loss", avg_W_loss, std_W_loss])
    writer.writerow(["W kval", avg_W_k, std_W_k])
    writer.writerow(["UW loss", avg_UW_loss, std_UW_loss])
    writer.writerow(["UW kval", avg_UW_k, std_UW_k])
# plt.figure()
# plt.scatter(data[:,0], data[:,1])
# plt.xlabel("Best k value")
# plt.ylabel("Minimum total weighted loss")
# plt.title("Best k value and associated minimum unweighted loss for 100 datasets \nof 100 events with 50% temporal density")
# plt.tight_layout()
# plt.savefig("0.5temp100x100/UWkval_vs_loss.png")

# plt.figure()
# plt.hist(data[:,0], bins = 11, edgecolor = 'black', alpha = 0.7)
# plt.xlabel("Best K value")
# plt.ylabel("Frequency")
# plt.title("Histogram of best unweighted k values for 100 datasets \nof 100 events with 50% temporal density")
# plt.savefig("0.5temp100x100/UWkval_hist.png")